In [3]:

!pip -q install datasets transformers evaluate scikit-learn torch accelerate matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


In [4]:
import os
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "TRUE"
os.environ["WANDB_DISABLED"] = "true"

import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [5]:
from datasets import load_dataset

dataset = load_dataset("glue", "qqp")

print(dataset)
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}


In [6]:
train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).rename("proportion"))
print(train_df["label"].value_counts().rename("count"))

train_df["q1_len"] = train_df["question1"].fillna("").str.split().str.len()
train_df["q2_len"] = train_df["question2"].fillna("").str.split().str.len()
print("\nQuestion length summary:")
print(train_df[["q1_len", "q2_len"]].describe())

Train size: 363846
Validation size: 40430

Train label distribution:
label
0    0.630673
1    0.369327
Name: proportion, dtype: float64
label
0    229468
1    134378
Name: count, dtype: int64

Question length summary:
              q1_len         q2_len
count  363846.000000  363846.000000
mean       10.942822      11.183468
std         5.437638       6.325961
min         1.000000       1.000000
25%         7.000000       7.000000
50%        10.000000      10.000000
75%        13.000000      13.000000
max       125.000000     237.000000


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def pair_to_text(df):
    return (df["question1"].fillna("") + " [SEP] " + df["question2"].fillna("")).tolist()

# Use a manageable subset for faster baseline training. Increase if you have time.
BASELINE_TRAIN_SIZE = 50000
baseline_train = train_df.sample(n=min(BASELINE_TRAIN_SIZE, len(train_df)), random_state=SEED)

X_train_base = pair_to_text(baseline_train)
y_train_base = baseline_train["label"].values
X_val_base = pair_to_text(val_df)
y_val = val_df["label"].values

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=100000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

baseline.fit(X_train_base, y_train_base)
baseline_preds = baseline.predict(X_val_base)

print("TF-IDF baseline results:")
print(classification_report(y_val, baseline_preds, digits=4))
print("Confusion matrix:", confusion_matrix(y_val, baseline_preds))

TF-IDF baseline results:
              precision    recall  f1-score   support

           0     0.8058    0.7932    0.7994     25545
           1     0.6544    0.6720    0.6630     14885

    accuracy                         0.7486     40430
   macro avg     0.7301    0.7326    0.7312     40430
weighted avg     0.7501    0.7486    0.7492     40430

Confusion matrix: [[20262  5283]
 [ 4883 10002]]


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["question1"],
        batch["question2"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["question1", "question2", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 390965
    })
})


In [9]:
TRAIN_SIZE = 30000
EVAL_SIZE = 5000

train_data = tokenized_dataset["train"].shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(tokenized_dataset["train"]))))
eval_data = tokenized_dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(tokenized_dataset["validation"]))))

print(train_data)
print(eval_data)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./bert-qqp-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [11]:

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch
import gc

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_experiment(
    model_name,
    max_length=128,
    learning_rate=2e-5,
    train_size=30000,
    eval_size=5000,
    epochs=3,
    batch_size=16
):
    print("=" * 80)
    print(f"Model: {model_name}")
    print(f"MAX_LENGTH: {max_length}")
    print(f"Learning rate: {learning_rate}")
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(example):
        return tokenizer(
            example["question1"],
            example["question2"],
            truncation=True,
            max_length=max_length
        )

    tokenized = dataset.map(tokenize_function, batched=True)

    train_data = tokenized["train"].shuffle(seed=SEED).select(
        range(min(train_size, len(tokenized["train"])))
    )

    eval_data = tokenized["validation"].shuffle(seed=SEED).select(
        range(min(eval_size, len(tokenized["validation"])))
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    output_dir = f"./results_{model_name.replace('/', '_')}_len{max_length}_lr{learning_rate}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )

    trainer.train()
    results = trainer.evaluate()

    # free GPU memory after experiment
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "model": model_name,
        "max_length": max_length,
        "learning_rate": learning_rate,
        "train_size": train_size,
        "eval_size": eval_size,
        "epochs": epochs,
        "accuracy": results["eval_accuracy"],
        "precision": results["eval_precision"],
        "recall": results["eval_recall"],
        "f1": results["eval_f1"],
    }

In [1]:
experiment_results = []

experiments = [
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "distilbert-base-uncased",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "roberta-base",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 64,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 256,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 1e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 3e-5,
    },
]

for exp in experiments:
    result = run_experiment(
        model_name=exp["model_name"],
        max_length=exp["max_length"],
        learning_rate=exp["learning_rate"],
        train_size=30000,
        eval_size=5000,
        epochs=3,
        batch_size=16
    )
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)
results_df = results_df.sort_values(by="f1", ascending=False)

display(results_df)

NameError: name 'run_experiment' is not defined

In [ ]:
bert_metrics = trainer.evaluate()
print(bert_metrics)

In [ ]:
pred_output = trainer.predict(eval_data)
logits = pred_output.predictions
labels = pred_output.label_ids
preds = np.argmax(logits, axis=-1)

print(classification_report(labels, preds, target_names=["not_paraphrase", "paraphrase"], digits=4))
print("Confusion matrix:", confusion_matrix(labels, preds))

In [ ]:
from scipy.special import softmax

probs = softmax(logits, axis=1)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for threshold in thresholds:
    tuned_preds = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, tuned_preds, average="binary", zero_division=0)
    rows.append({"threshold": threshold, "precision": precision, "recall": recall, "f1": f1})

threshold_df = pd.DataFrame(rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

best_threshold = threshold_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
print("Best threshold:", best_threshold)

In [ ]:
# Build a readable validation dataframe aligned with eval_data selection.
eval_size = len(eval_data)
raw_eval_sample = dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(dataset["validation"]))))
errors_df = pd.DataFrame(raw_eval_sample)
errors_df["pred"] = preds
errors_df["paraphrase_probability"] = probs
errors_df["error_type"] = np.where(
    (errors_df["label"] == 0) & (errors_df["pred"] == 1), "false_positive",
    np.where((errors_df["label"] == 1) & (errors_df["pred"] == 0), "false_negative", "correct")
)

print("False positives:")
display(errors_df[errors_df["error_type"] == "false_positive"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))

print("False negatives:")
display(errors_df[errors_df["error_type"] == "false_negative"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))